# Quick-Tuning: EWMA / Bollinger / GARCH Pairtrading

Bedient die drei schnellen Walk-Forward-Tuner (`quick_tune_ewma.py`,
`quick_tune_bollinger.py`, `quick_tune_garch.py`) ueber je ein eigenes
Widget-Panel -- ein Abschnitt pro Strategie.

Pro Parameter ist EINZELN waehlbar, ob er
  - **fix** auf einen Wert gesetzt wird (Checkbox "tunen?" AUS, Fix-Feld
    aktiv), oder
  - **gesucht** wird, mit einem einstellbaren Bereich (Checkbox "tunen?"
    AN, Bereichs-Feld aktiv) statt des im Tuner hart codierten
    Default-Bereichs.

Alle Voreinstellungen sind auf ein ZUEGIGES, GROBES Tuning ausgelegt
(kurzer Zeitraum, wenige Trials, wenige Fenster) -- fuer ein praezises
Tuning einfach Zeitraum/Dauer/Trials/Fenster in den Widgets hochsetzen.

Fenstergroessen (train_days/val_days/step_days/embargo_days) werden NICHT
einzeln eingestellt -- sie werden automatisch aus dem gewaehlten Zeitraum
und der einzigen echten Stellschraube `n_windows` (Anzahl Walk-Forward-
Fenster) abgeleitet, siehe `tuning_common.auto_walkforward_layout`.

Jeder Tuning-Lauf speichert automatisch:
  - die besten Parameter nach `data/processed/tuning/best_params_<strategie>.json`
  - einen `BEST_PARAMS`-Referenzblock direkt im jeweiligen
    `*_pairtrading_engine.py`-Modul

Zusaetzlich wird beim Start jedes Tuning-Laufs automatisch ein
`optuna-dashboard` fuer die jeweilige Study gestartet (falls noch nicht
laufend) und im Browser geoeffnet -- EWMA auf Port 8080, Bollinger auf
8081, GARCH auf 8082 (siehe `tuning_common.DASHBOARD_PORTS` und
`ensure_dashboard_running`). Alle drei koennen gleichzeitig laufen, ein
bereits laufendes Dashboard wird nicht doppelt gestartet.

`strategy_comparison.ipynb` liest diese gespeicherten besten Parameter
anschliessend automatisch fuer den finalen Out-of-Sample-Vergleich.

In [15]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))
sys.path.insert(0, str(Path("../..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import ipywidgets as widgets
import pandas as pd
from IPython.display import clear_output, display

import pairtrading_engine as pte
import bollinger_pairtrading_engine as bpe
import garch_pairtrading_engine as ge
import quick_tune_ewma as qte
import quick_tune_bollinger as qtb
import quick_tune_garch as qtg
import tuning_common as tc

from Src.config import (
    ASSET_ID_COL, ASSETS_PATH, CLIQUES_DIR, CLOSE_COL,
    MARKET_SYMBOL_COL, OHLCV_DIR, TIMESTAMP_COL,
)
from Src.clustering.helpers import STABLECOIN_SYMBOLS

print("Imports OK")

Imports OK


## 1. Daten laden (Cliquen + OHLCV + Asset-Symbole)

Einmal fuer alle drei Strategie-Panels gemeinsam geladen.

In [16]:
CLIQUES_VARIANT = "cliques_raw_12h"  # siehe data/processed/cliques/*.parquet

assets_df = pd.read_parquet(ASSETS_PATH)[["asset_id", "symbol"]].drop_duplicates()
id_to_symbol = dict(zip(assets_df["asset_id"], assets_df["symbol"]))
excluded_ids_stablecoins = set(
    assets_df.loc[assets_df["symbol"].isin(STABLECOIN_SYMBOLS), "asset_id"].tolist()
)

ohlcv_files = sorted(OHLCV_DIR.glob("*.parquet"))
raw = pd.concat([pd.read_parquet(f) for f in ohlcv_files], ignore_index=True)
raw["base_symbol"] = raw[MARKET_SYMBOL_COL].str.replace("USDT$", "", regex=True)
raw = raw.merge(assets_df.rename(columns={"symbol": "base_symbol"}), on="base_symbol", how="left")
raw = raw.dropna(subset=["asset_id"])
raw["asset_id"] = raw["asset_id"].astype(int)
raw[TIMESTAMP_COL] = pd.to_datetime(raw[TIMESTAMP_COL], utc=True)
raw = raw.loc[~raw["asset_id"].isin(excluded_ids_stablecoins)]
ohlcv_df = raw[[TIMESTAMP_COL, ASSET_ID_COL, CLOSE_COL]].copy()

cliques_df_full = pd.read_parquet(CLIQUES_DIR / f"{CLIQUES_VARIANT}.parquet")
basket_labels = sorted(cliques_df_full["basket"].unique())

data_min = ohlcv_df[TIMESTAMP_COL].min()
data_max = ohlcv_df[TIMESTAMP_COL].max()

print(f"Verfuegbarer Zeitraum: {data_min.date()} -> {data_max.date()}")
print(f"OHLCV: {ohlcv_df.shape[0]:,} Zeilen | Assets: {ohlcv_df[ASSET_ID_COL].nunique()}")
print(f"Baskets in '{CLIQUES_VARIANT}': {basket_labels}")

Verfuegbarer Zeitraum: 2024-07-23 -> 2026-07-22
OHLCV: 8,553,834 Zeilen | Assets: 45
Baskets in 'cliques_raw_12h': ['Basket 1', 'Basket 2', 'Basket 3', 'Basket 4']


## 2. Gemeinsame Widget-Bausteine

`param_row` baut pro Parameter eine Zeile mit Checkbox ("tunen?") +
Fix-Wert-Feld (aktiv wenn AUS) + Bereichs-Feld (aktiv wenn AN). Ein
Parameter ohne eigene Zeile bleibt im Default-Suchraum des jeweiligen
Tuners (siehe dortige Docstrings) -- diese Panels decken die gaengigsten
Stellschrauben ab, nicht zwingend jeden einzelnen Suchparameter.

In [17]:
def make_toggle_range_row(
    name: str, *, kind: str, fix_default, range_default, description: str,
    fix_min=None, fix_max=None, fix_step=None, range_min=None, range_max=None, range_step=None,
    options=None,
):
    """Baut (tune_checkbox, fix_widget, range_widget, row_box) fuer EINEN
    Parameter. kind: 'int', 'float' oder 'categorical'.

    - tune_checkbox AUS (Default) -> fix_widget aktiv, Parameter wird fixiert.
    - tune_checkbox AN -> range_widget aktiv, Parameter wird im gewaehlten
      Bereich gesucht (kind='categorical': SelectMultiple statt Range).
    """
    tune_checkbox = widgets.Checkbox(value=False, description="tunen?", indent=False, layout=widgets.Layout(width="90px"))

    if kind == "int":
        fix_widget = widgets.IntSlider(value=fix_default, min=fix_min, max=fix_max, step=fix_step or 1, description=name + ":", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"))
        range_widget = widgets.IntRangeSlider(value=range_default, min=range_min, max=range_max, step=range_step or 1, description=name + " (Bereich):", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"), disabled=True)
    elif kind == "float":
        fix_widget = widgets.FloatSlider(value=fix_default, min=fix_min, max=fix_max, step=fix_step or 0.01, description=name + ":", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"))
        range_widget = widgets.FloatRangeSlider(value=range_default, min=range_min, max=range_max, step=range_step or 0.01, description=name + " (Bereich):", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"), disabled=True)
    elif kind == "categorical":
        fix_widget = widgets.Dropdown(options=options, value=fix_default, description=name + ":", style={"description_width": "170px"})
        range_widget = widgets.SelectMultiple(options=options, value=tuple(range_default), description=name + " (Optionen):", style={"description_width": "170px"}, layout=widgets.Layout(width="420px", height="55px"), disabled=True)
    else:
        raise ValueError(f"Unbekannte kind={kind!r}")

    fix_widget.disabled = False

    def _on_toggle(change):
        range_widget.disabled = not change["new"]
        fix_widget.disabled = change["new"]

    tune_checkbox.observe(_on_toggle, names="value")

    desc_html = widgets.HTML(f"<span style='color:#888; font-size:0.85em;'>{description}</span>", layout=widgets.Layout(margin="8px 0 0 10px"))
    row = widgets.HBox([tune_checkbox, fix_widget, range_widget, desc_html])
    return tune_checkbox, fix_widget, range_widget, kind, row


def collect_fixed_and_search(param_widgets: dict) -> tuple[dict, dict]:
    """Liest alle param_row-Widgets aus und baut (fixed_params, search_ranges)
    fuer den Tuner-Aufruf -- Parameter mit tune_checkbox=AUS landen in
    fixed_params, Parameter mit tune_checkbox=AN in search_ranges (mit dem
    im Widget gewaehlten Bereich statt des Tuner-Default-Bereichs)."""
    fixed_params, search_ranges = {}, {}
    for name, (tune_checkbox, fix_widget, range_widget, kind, _row) in param_widgets.items():
        if tune_checkbox.value:
            search_ranges[name] = list(range_widget.value) if kind == "categorical" else tuple(range_widget.value)
        else:
            fixed_params[name] = fix_widget.value
    return fixed_params, search_ranges


print("Widget-Bausteine bereit.")

Widget-Bausteine bereit.


## 3. EWMA-Toleranzintervall-Strategie

In [18]:
ewma_w_basket = widgets.Dropdown(options=basket_labels, value=basket_labels[0], description="Basket:", style={"description_width": "170px"})
ewma_w_start = widgets.DatePicker(description="Start:", style={"description_width": "170px"})
ewma_w_total_days = widgets.IntSlider(value=30, min=10, max=365, step=5, description="Gesamtdauer (Tage):", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"))
ewma_w_test_fraction = widgets.FloatSlider(value=0.2, min=0.05, max=0.5, step=0.05, description="Test-Anteil:", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"))
ewma_w_windows = widgets.IntSlider(value=3, min=1, max=10, step=1, description="n_windows:", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"))
ewma_w_trials = widgets.IntSlider(value=20, min=5, max=200, step=5, description="n_trials:", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"))
ewma_w_fee = widgets.Dropdown(options=["standard", "bnb_discount", "vip1"], value="standard", description="Fee-Preset:", style={"description_width": "170px"})

ewma_w_start.value = max(pd.Timestamp(data_min), pd.Timestamp(data_max) - pd.Timedelta(days=60)).date()

ewma_params = {
    "lookback_candles": make_toggle_range_row("lookback_candles", kind="int", fix_default=87, range_default=(40, 140), description="Toleranzgrenzen-Lookback (Kerzen).", fix_min=20, fix_max=200, range_min=20, range_max=200),
    "p_content": make_toggle_range_row("p_content", kind="float", fix_default=0.95, range_default=(0.85, 0.98), description="Geforderter abgedeckter Anteil (Wilks).", fix_min=0.80, fix_max=0.99, fix_step=0.01, range_min=0.80, range_max=0.99, range_step=0.01),
    "ewma_decay": make_toggle_range_row("ewma_decay", kind="float", fix_default=0.94, range_default=(0.85, 0.97), description="EWMA-Decay der Vola-Normierung.", fix_min=0.80, fix_max=0.995, fix_step=0.005, range_min=0.80, range_max=0.995, range_step=0.005),
    "exit_lookback_candles": make_toggle_range_row("exit_lookback_candles", kind="int", fix_default=40, range_default=(20, 80), description="Fenster fuer den Exit-Mittelwert des Preis-Spreads.", fix_min=5, fix_max=150, range_min=5, range_max=150),
    "recalib_every": make_toggle_range_row("recalib_every", kind="int", fix_default=12, range_default=(6, 24), description="Rekalibrierungs-Intervall (Kerzen).", fix_min=1, fix_max=96, range_min=1, range_max=96),
    "max_holding_hours": make_toggle_range_row("max_holding_hours", kind="float", fix_default=12.0, range_default=(4.0, 24.0), description="Maximale Haltedauer (Stunden).", fix_min=1.0, fix_max=96.0, fix_step=0.5, range_min=1.0, range_max=96.0, range_step=0.5),
    "max_concurrent_positions": make_toggle_range_row("max_concurrent_positions", kind="int", fix_default=10, range_default=(3, 15), description="Max. gleichzeitig offene Positionen.", fix_min=1, fix_max=25, range_min=1, range_max=25),
    "trade_mode": make_toggle_range_row("trade_mode", kind="categorical", fix_default="max_only", range_default=["max_only", "all_significant"], description="'max_only' (Paper-Original) oder 'all_significant'.", options=["max_only", "all_significant"]),
}

ewma_run_button = widgets.Button(description="EWMA-Tuning starten", button_style="primary", icon="bolt")
ewma_output = widgets.Output()


def _ewma_on_run(_):
    with ewma_output:
        clear_output(wait=True)
        start_ts = pd.Timestamp(ewma_w_start.value, tz="UTC")
        end_ts = min(start_ts + pd.Timedelta(days=ewma_w_total_days.value), pd.Timestamp(data_max))
        train_val_start, train_val_end, test_start, test_end = tc.split_walkforward_test(
            start_ts, end_ts, test_fraction=ewma_w_test_fraction.value,
        )
        print(f"Zeitraum: {start_ts.date()} -> {end_ts.date()} | "
              f"Walk-Forward: {train_val_start.date()} -> {train_val_end.date()} | "
              f"Test: {test_start.date()} -> {test_end.date()}")

        fixed_params, search_ranges = collect_fixed_and_search(ewma_params)
        basket = ewma_w_basket.value
        cliques_df = cliques_df_full[cliques_df_full["basket"] == basket]

        result = qte.tune_ewma_strategy(
            basket=basket, cliques_df=cliques_df, ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
            timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
            train_val_start=train_val_start, train_val_end=train_val_end,
            test_start=test_start, test_end=test_end,
            n_windows=ewma_w_windows.value, n_trials=ewma_w_trials.value,
            fee_preset=ewma_w_fee.value, fixed_params=fixed_params, search_ranges=search_ranges,
        )
        print(f"\nFertig. Bestes Objective: {result.best_objective:.4f}")
        print(f"Beste Parameter: {result.best_params}")
        print(f"Gespeichert nach: {tc.best_params_path('ewma')}")


# _click_handlers.callbacks leeren, BEVOR on_click neu registriert wird --
# ohne das haengt sich bei jeder erneuten Ausfuehrung dieser Zelle (z.B.
# "Run All" nach einer Aenderung weiter oben) ein zusaetzlicher Handler an
# denselben Button, sodass EIN Klick mehrere parallele Tuning-Laeufe auf
# derselben Study auslöst (sichtbar an mehrfach durchmischten Log-Zeilen).
ewma_run_button._click_handlers.callbacks.clear()
ewma_run_button.on_click(_ewma_on_run)
display(widgets.VBox([
    widgets.HTML("<b>EWMA-Toleranzintervall (pairtrading_engine.py)</b>"),
    widgets.HTML("<u>Universum & Zeitraum</u>"),
    ewma_w_basket, ewma_w_start, ewma_w_total_days, ewma_w_test_fraction, ewma_w_windows, ewma_w_trials, ewma_w_fee,
    widgets.HTML("<u>Parameter</u> (Haekchen 'tunen?' AN = Suche im Bereichsfeld, AUS = fix im linken Feld)"),
    *[p[4] for p in ewma_params.values()],
    ewma_run_button, ewma_output,
]))

## 4. Bollinger-Band-Strategie

In [19]:
boll_w_basket = widgets.Dropdown(options=basket_labels, value=basket_labels[0], description="Basket:", style={"description_width": "170px"})
boll_w_start = widgets.DatePicker(description="Start:", style={"description_width": "170px"})
boll_w_total_days = widgets.IntSlider(value=30, min=10, max=365, step=5, description="Gesamtdauer (Tage):", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"))
boll_w_test_fraction = widgets.FloatSlider(value=0.2, min=0.05, max=0.5, step=0.05, description="Test-Anteil:", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"))
boll_w_windows = widgets.IntSlider(value=3, min=1, max=10, step=1, description="n_windows:", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"))
boll_w_trials = widgets.IntSlider(value=20, min=5, max=200, step=5, description="n_trials:", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"))
boll_w_fee = widgets.Dropdown(options=["standard", "bnb_discount", "vip1"], value="standard", description="Fee-Preset:", style={"description_width": "170px"})

boll_w_start.value = max(pd.Timestamp(data_min), pd.Timestamp(data_max) - pd.Timedelta(days=60)).date()

boll_params = {
    "lookback_candles": make_toggle_range_row("lookback_candles", kind="int", fix_default=60, range_default=(20, 100), description="Bollinger-Band-Lookback (Kerzen).", fix_min=20, fix_max=144, range_min=20, range_max=144),
    "n_std": make_toggle_range_row("n_std", kind="float", fix_default=2.0, range_default=(1.5, 3.0), description="Bandbreite in Standardabweichungen.", fix_min=1.5, fix_max=3.5, fix_step=0.1, range_min=1.5, range_max=3.5, range_step=0.1),
    "exit_lookback_candles": make_toggle_range_row("exit_lookback_candles", kind="int", fix_default=200, range_default=(100, 400), description="Fenster fuer den Exit-Mittelwert des Preis-Spreads.", fix_min=100, fix_max=1000, range_min=100, range_max=1000),
    "hedge_lookback_candles": make_toggle_range_row("hedge_lookback_candles", kind="int", fix_default=120, range_default=(60, 180), description="Fenster fuer die OLS-Hedge-Ratio-Schaetzung.", fix_min=60, fix_max=240, range_min=60, range_max=240),
    "max_holding_hours": make_toggle_range_row("max_holding_hours", kind="float", fix_default=12.0, range_default=(4.0, 24.0), description="Maximale Haltedauer (Stunden).", fix_min=4.0, fix_max=48.0, fix_step=0.5, range_min=4.0, range_max=48.0, range_step=0.5),
    "max_concurrent_positions": make_toggle_range_row("max_concurrent_positions", kind="int", fix_default=10, range_default=(3, 15), description="Max. gleichzeitig offene Positionen.", fix_min=3, fix_max=15, range_min=3, range_max=15),
    "trade_mode": make_toggle_range_row("trade_mode", kind="categorical", fix_default="max_only", range_default=["max_only", "all_significant"], description="'max_only' oder 'all_significant'.", options=["max_only", "all_significant"]),
}

boll_run_button = widgets.Button(description="Bollinger-Tuning starten", button_style="primary", icon="bolt")
boll_output = widgets.Output()


def _boll_on_run(_):
    with boll_output:
        clear_output(wait=True)
        start_ts = pd.Timestamp(boll_w_start.value, tz="UTC")
        end_ts = min(start_ts + pd.Timedelta(days=boll_w_total_days.value), pd.Timestamp(data_max))
        train_val_start, train_val_end, test_start, test_end = tc.split_walkforward_test(
            start_ts, end_ts, test_fraction=boll_w_test_fraction.value,
        )
        print(f"Zeitraum: {start_ts.date()} -> {end_ts.date()} | "
              f"Walk-Forward: {train_val_start.date()} -> {train_val_end.date()} | "
              f"Test: {test_start.date()} -> {test_end.date()}")

        fixed_params, search_ranges = collect_fixed_and_search(boll_params)
        basket = boll_w_basket.value
        cliques_df = cliques_df_full[cliques_df_full["basket"] == basket]

        result = qtb.tune_bollinger_strategy(
            basket=basket, cliques_df=cliques_df, ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
            timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
            train_val_start=train_val_start, train_val_end=train_val_end,
            test_start=test_start, test_end=test_end,
            n_windows=boll_w_windows.value, n_trials=boll_w_trials.value,
            fee_preset=boll_w_fee.value, fixed_params=fixed_params, search_ranges=search_ranges,
        )
        print(f"\nFertig. Bestes Objective: {result.best_objective:.4f}")
        print(f"Beste Parameter: {result.best_params}")
        print(f"Gespeichert nach: {tc.best_params_path('bollinger')}")


boll_run_button._click_handlers.callbacks.clear()
boll_run_button.on_click(_boll_on_run)
display(widgets.VBox([
    widgets.HTML("<b>Bollinger-Baender (bollinger_pairtrading_engine.py)</b>"),
    widgets.HTML("<u>Universum & Zeitraum</u>"),
    boll_w_basket, boll_w_start, boll_w_total_days, boll_w_test_fraction, boll_w_windows, boll_w_trials, boll_w_fee,
    widgets.HTML("<u>Parameter</u> (Haekchen 'tunen?' AN = Suche im Bereichsfeld, AUS = fix im linken Feld)"),
    *[p[4] for p in boll_params.values()],
    boll_run_button, boll_output,
]))

## 5. GARCH-Toleranzintervall-Strategie

Deutlich teurer pro Trial (GARCH(1,1)-Fit pro Paar/Tag) -- Default-Werte
fuer n_windows/n_trials daher niedriger als bei EWMA/Bollinger.

In [20]:
garch_w_basket = widgets.Dropdown(options=basket_labels, value=basket_labels[0], description="Basket:", style={"description_width": "170px"})
garch_w_start = widgets.DatePicker(description="Start:", style={"description_width": "170px"})
garch_w_total_days = widgets.IntSlider(value=30, min=10, max=365, step=5, description="Gesamtdauer (Tage):", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"))
garch_w_test_fraction = widgets.FloatSlider(value=0.2, min=0.05, max=0.5, step=0.05, description="Test-Anteil:", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"))
garch_w_windows = widgets.IntSlider(value=2, min=1, max=6, step=1, description="n_windows:", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"))
garch_w_trials = widgets.IntSlider(value=8, min=3, max=60, step=1, description="n_trials:", style={"description_width": "170px"}, layout=widgets.Layout(width="420px"))
garch_w_fee = widgets.Dropdown(options=["standard", "bnb_discount", "vip1"], value="standard", description="Fee-Preset:", style={"description_width": "170px"})

garch_w_start.value = max(pd.Timestamp(data_min), pd.Timestamp(data_max) - pd.Timedelta(days=60)).date()

garch_params = {
    "lookback_candles": make_toggle_range_row("lookback_candles", kind="int", fix_default=100, range_default=(60, 140), description="Toleranzgrenzen-Lookback (Kerzen).", fix_min=60, fix_max=160, range_min=60, range_max=160),
    "p_content": make_toggle_range_row("p_content", kind="categorical", fix_default=0.95, range_default=[0.90, 0.95], description="Geforderter abgedeckter Anteil (Wilks).", options=[0.90, 0.95]),
    "recalib_every": make_toggle_range_row("recalib_every", kind="int", fix_default=12, range_default=(6, 24), description="Rekalibrierungs-Intervall (Kerzen).", fix_min=6, fix_max=48, range_min=6, range_max=48),
    "max_holding_hours": make_toggle_range_row("max_holding_hours", kind="float", fix_default=12.0, range_default=(4.0, 24.0), description="Maximale Haltedauer (Stunden).", fix_min=4.0, fix_max=48.0, fix_step=0.5, range_min=4.0, range_max=48.0, range_step=0.5),
    "max_concurrent_positions": make_toggle_range_row("max_concurrent_positions", kind="int", fix_default=10, range_default=(3, 15), description="Max. gleichzeitig offene Positionen.", fix_min=3, fix_max=15, range_min=3, range_max=15),
    "trade_mode": make_toggle_range_row("trade_mode", kind="categorical", fix_default="max_only", range_default=["max_only", "all_significant"], description="'max_only' oder 'all_significant'.", options=["max_only", "all_significant"]),
}

garch_run_button = widgets.Button(description="GARCH-Tuning starten", button_style="primary", icon="bolt")
garch_output = widgets.Output()


def _garch_on_run(_):
    with garch_output:
        clear_output(wait=True)
        start_ts = pd.Timestamp(garch_w_start.value, tz="UTC")
        end_ts = min(start_ts + pd.Timedelta(days=garch_w_total_days.value), pd.Timestamp(data_max))
        train_val_start, train_val_end, test_start, test_end = tc.split_walkforward_test(
            start_ts, end_ts, test_fraction=garch_w_test_fraction.value,
        )
        print(f"Zeitraum: {start_ts.date()} -> {end_ts.date()} | "
              f"Walk-Forward: {train_val_start.date()} -> {train_val_end.date()} | "
              f"Test: {test_start.date()} -> {test_end.date()}")

        fixed_params, search_ranges = collect_fixed_and_search(garch_params)
        basket = garch_w_basket.value
        cliques_df = cliques_df_full[cliques_df_full["basket"] == basket]

        result = qtg.tune_garch_strategy(
            basket=basket, cliques_df=cliques_df, ohlcv_df=ohlcv_df, id_to_symbol=id_to_symbol,
            timestamp_column=TIMESTAMP_COL, asset_id_column=ASSET_ID_COL, close_column=CLOSE_COL,
            train_val_start=train_val_start, train_val_end=train_val_end,
            test_start=test_start, test_end=test_end,
            n_windows=garch_w_windows.value, n_trials=garch_w_trials.value,
            fee_preset=garch_w_fee.value, fixed_params=fixed_params, search_ranges=search_ranges,
        )
        print(f"\nFertig. Bestes Objective: {result.best_objective:.4f}")
        print(f"Beste Parameter: {result.best_params}")
        print(f"Gespeichert nach: {tc.best_params_path('garch')}")


garch_run_button._click_handlers.callbacks.clear()
garch_run_button.on_click(_garch_on_run)
display(widgets.VBox([
    widgets.HTML("<b>GARCH-Toleranzintervall (garch_pairtrading_engine.py)</b>"),
    widgets.HTML("<u>Universum & Zeitraum</u>"),
    garch_w_basket, garch_w_start, garch_w_total_days, garch_w_test_fraction, garch_w_windows, garch_w_trials, garch_w_fee,
    widgets.HTML("<u>Parameter</u> (Haekchen 'tunen?' AN = Suche im Bereichsfeld, AUS = fix im linken Feld)"),
    *[p[4] for p in garch_params.values()],
    garch_run_button, garch_output,
]))

## 6. Uebersicht: aktuell gespeicherte beste Parameter je Strategie/Basket

Zeigt den zentralen Stand aus `data/processed/tuning/best_params_*.json` --
`strategy_comparison.ipynb` verwendet genau diese Werte fuer den finalen
Out-of-Sample-Vergleich.

In [21]:
def show_best_params_overview():
    rows = []
    for strategy_key in ("ewma", "bollinger", "garch"):
        for basket, params in tc.load_best_params(strategy_key).items():
            rows.append({
                "strategy": strategy_key, "basket": basket,
                "objective": params.get("objective"), "n_trials": params.get("n_trials"),
                "tuned_at": params.get("tuned_at"),
            })
    overview_df = pd.DataFrame(rows)
    if overview_df.empty:
        print("Noch keine getunten Parameter gespeichert -- oben mindestens ein Tuning-Panel ausfuehren.")
    else:
        display(overview_df.sort_values(["basket", "strategy"]))
    return overview_df


_ = show_best_params_overview()

,strategy,basket,objective,n_trials,tuned_at
0,ewma,Basket 2,-0.029764,30,2026-08-13T16:17:56.715377+00:00
1,garch,Basket 2,-0.065846,2,2026-08-13T16:08:44.113823+00:00
